# Exercice 20b (cloud, facultatif) : le mini-GPT en taille nanoGPT

Même modèle que l'exercice 20a (Transformer décodeur seul, pré-normalisé, caractère par caractère sur TinyShakespeare), mais dans la configuration `train_shakespeare_char` de nanoGPT : **6 couches, 6 têtes, 384 dimensions, contexte 256, dropout 0,2, 5 000 pas** (≈ 10,7 millions de paramètres). Sur un A100, le dépôt nanoGPT annonce environ 3 minutes et une perte de validation de 1,4697 ; sur le M1 Pro, la phase 1 du cours a mesuré environ 56 minutes. Ici : un GPU T4 gratuit (Colab ou Kaggle).

**Perte optimisée** : l'entropie croisée du token suivant ; perplexité = exp(perte).

Avant de lancer : *Exécution › Modifier le type d'exécution › GPU T4*, puis *Exécution › Tout exécuter*. Rien à installer : torch, numpy et matplotlib sont déjà dans l'image Colab. Seed 42. Calcul en **float16** (autocast + GradScaler) : le T4 ne fait pas de bfloat16.

In [ ]:
# Cellule 1 : vérifier le GPU
!nvidia-smi
import torch
if not torch.cuda.is_available():
    raise SystemExit("Pas de GPU : Exécution › Modifier le type d'exécution › GPU T4, puis relancer.")
print("GPU :", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# Cellule 2 : configuration (valeurs de nanoGPT, config/train_shakespeare_char.py)
import math, time, random, urllib.request
from dataclasses import dataclass, asdict
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda")

# COURT = True : 500 pas seulement, pour vérifier que tout tourne (quelques minutes)
COURT = False
config = dict(
    n_couches=6, n_tetes=6, n_embd=384, taille_bloc=256, dropout=0.2,
    taille_lot=64, n_pas=500 if COURT else 5000,
    lr_max=1e-3, lr_min=1e-4, pas_chauffe=100,
    eval_tous=250, n_lots_eval=50,   # nanoGPT : 200 lots d'évaluation ; 50 suffisent et vont 4 fois plus vite
)
print(config)

In [ ]:
# Cellule 3 : données (TinyShakespeare, 1,1 Mo) et tokeniseur caractère par caractère
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
texte = urllib.request.urlopen(URL).read().decode("utf-8")
caracteres = sorted(set(texte))
vers_id = {c: i for i, c in enumerate(caracteres)}
ids = torch.tensor([vers_id[c] for c in texte], dtype=torch.long)
n = int(0.9 * len(ids))
ids_train, ids_val = ids[:n], ids[n:]
print(f"{len(texte)} caractères, vocabulaire {len(caracteres)}, train {len(ids_train)}, val {len(ids_val)}")

def lot_aleatoire(donnees, taille_bloc, taille_lot, generateur=None):
    debuts = torch.randint(len(donnees) - taille_bloc - 1, (taille_lot,), generator=generateur)
    x = torch.stack([donnees[i:i + taille_bloc] for i in debuts])
    y = torch.stack([donnees[i + 1:i + 1 + taille_bloc] for i in debuts])
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)

In [ ]:
# Cellule 4 : le modèle, copie de ia_commun/mini_gpt.py (mêmes noms de poids : le checkpoint
# se recharge en local avec ia_commun.mini_gpt.MiniGPT.charger)
@dataclass
class ConfigGPT:
    taille_vocab: int = 65
    taille_bloc: int = 128
    n_couches: int = 4
    n_tetes: int = 4
    n_embd: int = 256
    dropout: float = 0.1
    biais: bool = False

class AttentionCausale(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_tetes = cfg.n_tetes
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=cfg.biais)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.biais)
        self.dropout = cfg.dropout
        self.drop_residuel = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_tetes, C // self.n_tetes).transpose(1, 2) for t in (q, k, v))
        # softmax(QKᵀ/√d_k)·V avec masque causal (noyau fusionné de PyTorch)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.drop_residuel(self.proj(y.transpose(1, 2).contiguous().view(B, T, C)))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=cfg.biais)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=cfg.biais)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))

class Bloc(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd, bias=cfg.biais)
        self.attn = AttentionCausale(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd, bias=cfg.biais)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))          # pré-normalisation
        return x + self.mlp(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.emb_tokens = nn.Embedding(cfg.taille_vocab, cfg.n_embd)
        self.emb_positions = nn.Embedding(cfg.taille_bloc, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocs = nn.ModuleList([Bloc(cfg) for _ in range(cfg.n_couches)])
        self.ln_final = nn.LayerNorm(cfg.n_embd, bias=cfg.biais)
        self.tete = nn.Linear(cfg.n_embd, cfg.taille_vocab, bias=False)
        self.tete.weight = self.emb_tokens.weight      # poids partagés
        self.apply(self._init_poids)
        for nom, p in self.named_parameters():
            if nom.endswith("proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_couches))

    @staticmethod
    def _init_poids(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx, cibles=None):
        T = idx.shape[1]
        x = self.drop(self.emb_tokens(idx) + self.emb_positions(torch.arange(T, device=idx.device)))
        for bloc in self.blocs:
            x = bloc(x)
        logits = self.tete(self.ln_final(x))
        loss = None if cibles is None else F.cross_entropy(logits.view(-1, logits.size(-1)).float(), cibles.view(-1))
        return logits, loss

cfg = ConfigGPT(taille_vocab=len(caracteres), taille_bloc=config["taille_bloc"], n_couches=config["n_couches"],
                n_tetes=config["n_tetes"], n_embd=config["n_embd"], dropout=config["dropout"])
modele = MiniGPT(cfg).to(device)
n_param = sum(p.numel() for p in modele.parameters()) - modele.emb_positions.weight.numel()
print(f"{n_param / 1e6:.2f} M paramètres (hors positions)")

In [ ]:
# Cellule 5 : entraînement en float16 (autocast + GradScaler), meilleur modèle gardé
def taux(pas):
    if pas < config["pas_chauffe"]:
        return config["lr_max"] * (pas + 1) / config["pas_chauffe"]
    r = min(1.0, (pas - config["pas_chauffe"]) / (config["n_pas"] - config["pas_chauffe"]))
    return config["lr_min"] + 0.5 * (config["lr_max"] - config["lr_min"]) * (1 + math.cos(math.pi * r))

@torch.no_grad()
def estimer(donnees, graine):
    modele.eval()
    g = torch.Generator().manual_seed(graine)
    pertes = []
    for _ in range(config["n_lots_eval"]):
        x, y = lot_aleatoire(donnees, config["taille_bloc"], config["taille_lot"], g)
        with torch.autocast("cuda", dtype=torch.float16):
            pertes.append(modele(x, y)[1].item())
    modele.train()
    return sum(pertes) / len(pertes)

optim = torch.optim.AdamW(modele.parameters(), lr=config["lr_max"], betas=(0.9, 0.99), weight_decay=0.1)
scaler = torch.amp.GradScaler("cuda")
gen = torch.Generator().manual_seed(SEED)
historique = {"pas": [], "train": [], "val": []}
meilleure, etat_meilleur = float("inf"), None
t0 = time.perf_counter()
for pas in range(config["n_pas"] + 1):
    if pas % config["eval_tous"] == 0 or pas == config["n_pas"]:
        tr, va = estimer(ids_train, 1), estimer(ids_val, 2)
        historique["pas"].append(pas); historique["train"].append(tr); historique["val"].append(va)
        if va < meilleure:
            meilleure = va
            etat_meilleur = {k: v.detach().cpu().clone() for k, v in modele.state_dict().items()}
        print(f"pas {pas:5d}  train {tr:.3f}  val {va:.3f}  ({time.perf_counter() - t0:.0f} s)")
    if pas == config["n_pas"]:
        break
    for g in optim.param_groups:
        g["lr"] = taux(pas)
    x, y = lot_aleatoire(ids_train, config["taille_bloc"], config["taille_lot"], gen)
    with torch.autocast("cuda", dtype=torch.float16):
        _, loss = modele(x, y)
    optim.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optim)
    torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)
    scaler.step(optim)
    scaler.update()
duree = time.perf_counter() - t0
print(f"Durée : {duree / 60:.1f} min ; meilleure val {meilleure:.3f} (perplexité {math.exp(meilleure):.2f})")
print("Notez cette durée : elle complète le README (durée T4 à mesurer).")
if not COURT and meilleure > 1.55:
    print("ATTENTION : val > 1,55, au-dessus du seuil annoncé par le README.")

In [ ]:
# Cellule 6 : courbes
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(historique["pas"], historique["train"], label="train")
plt.plot(historique["pas"], historique["val"], label="validation")
plt.xlabel("pas"); plt.ylabel("perte (nats par caractère)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Mini-GPT 6 × 384, TinyShakespeare")
plt.savefig("courbes-20b.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Cellule 7 : génération avec le meilleur modèle (température 0,8)
modele.load_state_dict(etat_meilleur)
modele.eval()
g = torch.Generator().manual_seed(SEED)
idx = torch.tensor([[vers_id["\n"]]], device=device)
with torch.no_grad():
    for _ in range(500):
        logits, _ = modele(idx[:, -config["taille_bloc"]:])
        p = F.softmax(logits[:, -1, :].float() / 0.8, dim=-1)
        idx = torch.cat([idx, torch.multinomial(p.cpu(), 1, generator=g).to(device)], dim=1)
print("".join(caracteres[i] for i in idx[0].tolist()))

In [ ]:
# Cellule 8 : sauvegarder AVANT la fin de session (même format que MiniGPT.sauvegarder)
chemin = "mini-gpt-shakespeare-grand.pt"
torch.save({"config": asdict(cfg), "etat": etat_meilleur, "caracteres": caracteres, "pas": config["n_pas"],
            "val_loss": meilleure, "historique": historique, "duree_s": duree}, chemin)
print("Checkpoint :", chemin)
try:
    from google.colab import files      # Colab : téléchargement direct
    files.download(chemin)
    files.download("courbes-20b.png")
except ImportError:
    print("Hors Colab (Kaggle) : le fichier est dans /kaggle/working, onglet Output.")